# PPD — Post-hoc Popularity Debiasing (baseline for comparison)

**Paper:** Md Aminul Islam, Elena Zheleva, Ren Wang. *"Post-hoc Popularity Bias
Correction in GNN-based Collaborative Filtering."* WWW '26 (arXiv:2510.12959).

This notebook reproduces PPD **on the exact same ML-1M data pipeline, LightGCN
backbone, train/val/test splits, exclusion sets, and evaluation protocol** as
your own post-hoc BCPS pipeline, so that the two post-hoc methods can be
compared head-to-head on a shared backbone.

* **CELL 1** and **CELL 2** below are copied *verbatim, unmodified* from your
  own notebook (`final-bpsd-2.ipynb`): same split construction
  (`build_splits` / `read_splits`), same balancing/recycling logic, same
  exclusion-set construction, same `LightGCN` backbone, same training loop,
  same `evaluate()` function, same `TOP_KS = [20, 50, 100]`, same tail
  definition. Nothing about the data or the backbone is changed.
* **CELL PPD-1 / CELL PPD-2** implement PPD (Algorithm 1 of the paper) as a
  *post-hoc* correction applied to the SAME trained backbone (`backbone`,
  `sparse_adj`), and evaluate it with your notebook's own `evaluate()`,
  `EXCL_VAL` / `EXCL_TEST`, `TAIL_VAL` / `TAIL_TEST`, and `item_pop`.

Run CELL 1 → CELL 2 (or reuse your existing kernel if `backbone`,
`sparse_adj`, `train_records`, etc. are already in memory) → CELL PPD-1 →
CELL PPD-2.


In [1]:
# ============================================================================
# CELL 1 — CONFIG, SPLITS, EXCLUSION SETS
# ============================================================================
import os, sys, time, math, json, copy, random, collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


# --- locate ml-1m ------------------------------------------------------------
# Only ratings.dat + movies.dat are ever read.
# Set this to the folder that CONTAINS ratings.dat and movies.dat.
ML1M_DIR = '/kaggle/input/datasets/tejasdeshmukh001/movielens'


def _find_raw_dir():
    base = ML1M_DIR or ''
    candidates = [
        base,
        os.path.join(base, 'ml-1m'),           # if the zip kept its ml-1m/ folder
        os.environ.get('ML1M_DIR'),
        '/kaggle/input/movielens-1m/ml-1m',
        '/kaggle/input/movielens-1m',
        './ml-1m', './data/ml-1m', '../ml-1m',
        os.path.expanduser('~/Documents/BPSD/ml-1m'),
    ]
    for c in candidates:
        if c and os.path.isfile(os.path.join(c, 'ratings.dat')) \
             and os.path.isfile(os.path.join(c, 'movies.dat')):
            return os.path.abspath(c)
    # Not found -> show what IS mounted so the right path can be pasted in.
    found = []
    if os.path.isdir('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if 'ratings.dat' in files:
                found.append(root)
    raise FileNotFoundError(
        'Could not find ml-1m.\n'
        + (f'Folders under /kaggle/input containing ratings.dat:\n  '
           + '\n  '.join(found) + '\n\nSet ML1M_DIR to one of these.'
           if found else
           'Nothing under /kaggle/input has a ratings.dat -- add the dataset via '
           '"+ Add Input" in the right-hand panel first.'))


RAW_DATA_DIR = _find_raw_dir()

# /kaggle/input is READ-ONLY, so every output goes to /kaggle/working.
OUT_ROOT    = ('/kaggle/working/bpsd_out' if os.path.isdir('/kaggle/working')
               else os.environ.get('BPSD_OUT', './bpsd_out'))
WORKING_DIR = os.path.join(OUT_ROOT, 'dataset', 'ml-1m')
CKPT_DIR    = os.path.join(OUT_ROOT, 'checkpoints')
os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'[paths] raw  = {RAW_DATA_DIR}')
print(f'[paths] out  = {os.path.abspath(OUT_ROOT)}')

SEED = 2020
POSITIVE_RATING_THRESHOLD = 4.0   # PPAC's "rating > 3". Their Table 1 reports
                                  # 6,038 users, which is EXACTLY the count of
                                  # ML-1M users having >=1 rating of 4+. Keep 4.0.
TEST_HOLDOUT_PER_USER     = 30    # PPAC's TOP_K
BALANCE_PER_ITEM          = None  # None -> derive from this split's own holdout pool
VAL_HOLDOUT_PER_USER      = 20    # only used when STRICT_PPAC = False

# STRICT_PPAC = True  -> the paper's released protocol: no validation split,
#                        checkpoint selected by NDCG@50 on the test set.
#                        Use ONLY for the head-to-head against Table 2.
# STRICT_PPAC = False -> carve a validation split and select on it. Honest.
STRICT_PPAC = False

# PPAC Sec 4.1: "...another 10% as the validation set using the same way.
# The remaining interactions are used for training."  i.e. PPAC has NO orphaned
# interactions -- anything not sampled into the balanced test/val is TRAINING
# data. Balancing here discards ~117k held-out positives; leaving them in limbo
# (neither trained on nor masked) is what inflated the old numbers.
RECYCLE_DISCARDS = True   # put balancing's discards back into train, per PPAC

EVAL_ON_BALANCED = True   # True  -> balanced (intervened) test set == paper's protocol
                          # False -> the raw 30-per-user holdout    == biased test set

TOP_KS        = [20, 50, 100]
TAIL_FRACTION = 1.0 / 3.0   # bottom third of the EVALUATED item universe, by train popularity
SAVE_CHECKPOINTS = False    # opt-in; nothing in this notebook ever READS a .pt
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[device] {DEVICE}')


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)


def build_splits():
    """Reproduce PPAC's ml-1M split, plus its balanced (intervened) test set.

    PPAC (run_MF.py::create_train_and_test):
      * positives are ratings > 3
      * every user with MORE than 30 positives contributes exactly 30 to test
      * every other user contributes all of their positives to train
      * NO global minimum-interaction user filter, NO validation split

    Balanced (intervened) set: fix a per-item quota n, keep every item with at
    least n held-out interactions, subsample exactly n. Every retained item then
    contributes the same number of test interactions.

    n is not a free constant. Retained = n * |{i : c_i >= n}| has an interior
    maximum: small n throws away interactions from popular items, large n throws
    away items entirely. BALANCE_PER_ITEM = None picks the argmax from THIS
    split's own item counts.

    Built entirely from ratings.dat and movies.dat. No external split files.
    """
    rng = random.Random(SEED)
    by_user = collections.defaultdict(list)
    rating_of, ts_of = {}, {}

    with open(os.path.join(RAW_DATA_DIR, 'ratings.dat'), encoding='latin-1') as fh:
        for line in fh:
            u, i, r, t = line.strip().split('::')
            u, r, t = int(u), float(r), int(t)
            if r >= POSITIVE_RATING_THRESHOLD:
                by_user[u].append(i)
                rating_of[(u, i)] = r
                ts_of[(u, i)] = t

    train, val, test_pool = (collections.defaultdict(list) for _ in range(3))
    for u, items in by_user.items():
        items = list(items)
        rng.shuffle(items)
        need = TEST_HOLDOUT_PER_USER + (0 if STRICT_PPAC else VAL_HOLDOUT_PER_USER)
        if len(items) > need:
            test_pool[u] = items[:TEST_HOLDOUT_PER_USER]
            if not STRICT_PPAC:
                val[u] = items[TEST_HOLDOUT_PER_USER:need]
            train[u] = items[need:]
        else:
            train[u] = items

    def _balance(pool, quota=None):
        hits = collections.defaultdict(list)
        for u, items in pool.items():
            for i in items:
                hits[i].append(u)
        counts = np.array(sorted(len(v) for v in hits.values()))
        retained = counts[::-1] * (np.arange(len(counts)) + 1)
        q = int(quota) if quota else int(counts[::-1][retained.argmax()])
        out, n_items = collections.defaultdict(list), 0
        for i, users in hits.items():
            if len(users) >= q:
                n_items += 1
                for u in rng.sample(users, q):
                    out[u].append(i)
        return out, q, n_items

    balanced, quota, n_bal_items = _balance(test_pool, BALANCE_PER_ITEM)
    if STRICT_PPAC:
        bal_val, vq, vn = collections.defaultdict(list), 0, 0
    else:
        bal_val, vq, vn = _balance(val)

    print(f'[split] users={len(by_user)}')
    print(f'[split] raw test: {len(test_pool)} users, '
          f'{sum(map(len, test_pool.values()))} interactions')
    print(f'[split] balanced test: {len(balanced)} users, '
          f'{sum(map(len, balanced.values()))} interactions over '
          f'{n_bal_items} items @ {quota} each')
    if not STRICT_PPAC:
        print(f'[split] raw val: {len(val)} users, {sum(map(len, val.values()))} interactions')
        print(f'[split] balanced val: {len(bal_val)} users, '
              f'{sum(map(len, bal_val.values()))} interactions over {vn} items @ {vq} each')

    # --- PPAC: "The remaining interactions are used for training." --------------
    # Balancing throws away held-out positives (~35/user). PPAC has no such
    # category: anything not sampled into the balanced test/val is training data.
    # Leaving them orphaned -- neither trained on nor masked at eval -- is what
    # made a vanilla LightGCN beat PPAC's own published method.
    n_before = sum(map(len, train.values()))
    if RECYCLE_DISCARDS:
        n_rec = 0
        for pool, keepset in ((test_pool, balanced), (val, bal_val)):
            for u, items in pool.items():
                k = set(keepset.get(u, []))
                extra = [i for i in items if i not in k]
                train[u].extend(extra)
                n_rec += len(extra)
        print(f'[split] recycled {n_rec:,} discarded holdout interactions into train '
              f'(PPAC Sec 4.1); train {n_before:,} -> {sum(map(len, train.values())):,}')
    else:
        orphan = (sum(map(len, test_pool.values())) - sum(map(len, balanced.values()))
                  + sum(map(len, val.values())) - sum(map(len, bal_val.values())))
        print(f'[split] WARNING: {orphan:,} held-out positives are in NO split '
              f'(not trained on, not scored). This is not PPAC\'s protocol.')
    print(f'[split] final train_inter={sum(map(len, train.values()))}')

    def dump(name, recs):
        with open(os.path.join(WORKING_DIR, name), 'w', encoding='utf-8') as fh:
            for u, items in recs.items():
                for i in items:
                    fh.write(f'{u}::{i}::{rating_of[(u, i)]:.1f}::{ts_of[(u, i)]}\n')

    dump('ratings.train', train)
    dump('ratings.test', test_pool)
    dump('balance_ratings.test', balanced)
    if not STRICT_PPAC:
        dump('ratings.val', val)
        dump('balance_ratings.val', bal_val)


def read_splits():
    """Load the splits and remap to compact 0-based ids.

    Returns SCORING targets (balanced, restricted to items seen in train) and,
    separately, the RAW holdout pools used only for masking. Keeping these
    apart is the fix for the exclusion bug: balancing discards held-out
    positives, and those discarded items must still be masked out of the
    candidate list even though they are never scored.
    """
    item_map = {}
    with open(os.path.join(RAW_DATA_DIR, 'movies.dat'), encoding='latin-1') as fh:
        for idx, line in enumerate(fh):
            item_map[line.split('::')[0]] = idx

    def load(name):
        recs = collections.defaultdict(list)
        path = os.path.join(WORKING_DIR, name)
        if not os.path.exists(path):
            return recs
        with open(path, encoding='utf-8') as fh:
            for line in fh:
                u, i, _, _ = line.strip().split('::')
                recs[int(u)].append(item_map[i])
        return recs

    raw_train = load('ratings.train')
    user_map = {raw: new for new, raw in enumerate(sorted(raw_train))}
    train = collections.defaultdict(list, {user_map[u]: v for u, v in raw_train.items()})
    train_items = {i for v in train.values() for i in v}

    def remap(name, rankable_only):
        """rankable_only=True  -> scoring target: drop items never seen in train.
           rankable_only=False -> masking pool: keep every known positive."""
        out = collections.defaultdict(list)
        for u, items in load(name).items():
            if u not in user_map:
                continue
            keep = items if not rankable_only else [i for i in items if i in train_items]
            if keep:
                out[user_map[u]] = keep
        return out

    test = remap('balance_ratings.test' if EVAL_ON_BALANCED else 'ratings.test', True)
    val  = remap('balance_ratings.val'  if EVAL_ON_BALANCED else 'ratings.val',  True)

    # FIX: masking pools come from the RAW holdouts, never the balanced subsets.
    raw_val  = remap('ratings.val',  False)   # empty dict when STRICT_PPAC
    raw_test = remap('ratings.test', False)

    if STRICT_PPAC:
        val = test          # paper protocol: selection happens on the eval set
    params = {'num_users': len(user_map), 'num_items': len(item_map)}
    return train, val, test, raw_val, raw_test, user_map, item_map, params


set_seed(SEED)
build_splits()
(train_records, val_records, test_records,
 raw_val_records, raw_test_records, user_map, item_map, params) = read_splits()
NUM_USERS, NUM_ITEMS = params['num_users'], params['num_items']


# --- exclusion = every known positive MINUS the current scoring target -------
# With RECYCLE_DISCARDS=True this is now just the STANDARD rule (mask the user's
# training items, plus val when scoring test) -- because the discards ARE
# training items. The ambiguity that existed before disappears once the split
# stops orphaning them.
def exclusion_for(target):
    """Mask everything the user is known to like, except what we score now."""
    out = {}
    for u in train_records:
        seen = (set(train_records.get(u, []))
                | set(raw_val_records.get(u, []))
                | set(raw_test_records.get(u, [])))
        out[u] = list(seen - set(target.get(u, [])))
    return out


EXCL_VAL  = exclusion_for(val_records)
EXCL_TEST = exclusion_for(test_records)

_per_user_old = np.mean([len(train_records.get(u, [])) + len(val_records.get(u, []))
                         for u in test_records])
_per_user_new = np.mean([len(EXCL_TEST.get(u, [])) for u in test_records])
print(f'[data] users={NUM_USERS} items={NUM_ITEMS} '
      f'train={sum(map(len, train_records.values()))} '
      f'val_users={len(val_records)} test_users={len(test_records)}')
print(f'[mask] mean masked items per test user: {_per_user_new:.1f}')
if RECYCLE_DISCARDS:
    print('[mask] == standard rule (train + val), since discards are now training data')

[paths] raw  = /kaggle/input/datasets/tejasdeshmukh001/movielens
[paths] out  = /kaggle/working/bpsd_out
[device] cuda
[split] users=6038
[split] raw test: 3329 users, 99870 interactions
[split] balanced test: 3329 users, 29882 interactions over 446 items @ 67 each
[split] raw val: 3329 users, 66580 interactions
[split] balanced val: 3325 users, 19932 interactions over 453 items @ 44 each
[split] recycled 116,636 discarded holdout interactions into train (PPAC Sec 4.1); train 408,831 -> 525,467
[split] final train_inter=525467
[data] users=6038 items=3883 train=525467 val_users=3325 test_users=3329
[mask] mean masked items per test user: 141.8
[mask] == standard rule (train + val), since discards are now training data


In [2]:
# ============================================================================
# CELL 2 — STAGE 1: LIGHTGCN BACKBONE  (LOADED FROM THE BPSD CHECKPOINT; never retrained)
# ============================================================================
LATENT_DIM = 64
N_LAYERS   = 3
LR         = 1e-3       # paper reports 0.01; 1e-3 is the released-code default
REG_WEIGHT = 1e-4
BATCH_SIZE = 8192
EPOCHS     = 2000
PATIENCE   = 50


def build_sparse_adj(train_recs, num_users, num_items):
    n = num_users + num_items
    us, it = [], []
    for u, items in train_recs.items():
        us.extend([u] * len(items))
        it.extend([i + num_users for i in items])
    us = torch.tensor(us, dtype=torch.long)
    it = torch.tensor(it, dtype=torch.long)
    src = torch.cat([us, it]); dst = torch.cat([it, us])
    deg = torch.zeros(n).scatter_add_(0, dst, torch.ones(dst.numel()))
    dis = deg.clamp(min=1.0).pow(-0.5)
    return torch.sparse_coo_tensor(
        torch.stack([dst, src]), dis[dst] * dis[src], (n, n)).coalesce()


def propagate(adj, e_u0, e_i0, n_layers=N_LAYERS):
    """LightGCN readout: mean over layers 0..L (alpha_l = 1/(L+1))."""
    x = torch.cat([e_u0, e_i0], dim=0)
    layers = [x]
    for _ in range(n_layers):
        x = torch.sparse.mm(adj, x)
        layers.append(x)
    out = torch.stack(layers, dim=1).mean(dim=1)
    return torch.split(out, [e_u0.shape[0], e_i0.shape[0]], dim=0)


def sample_triplets(train_recs, num_items):
    users, pos = [], []
    for u, items in train_recs.items():
        users.extend([u] * len(items)); pos.extend(items)
    users = np.asarray(users, dtype=np.int64); pos = np.asarray(pos, dtype=np.int64)
    psets = {u: set(v) for u, v in train_recs.items()}
    neg = np.random.randint(0, num_items, size=len(users), dtype=np.int64)
    bad = np.fromiter((n in psets[u] for u, n in zip(users, neg)), bool, len(users))
    while bad.any():
        idx = np.flatnonzero(bad)
        neg[idx] = np.random.randint(0, num_items, size=len(idx), dtype=np.int64)
        bad[idx] = np.fromiter((neg[j] in psets[users[j]] for j in idx), bool, len(idx))
    return torch.from_numpy(users), torch.from_numpy(pos), torch.from_numpy(neg)


def recall_ndcg(ground_truth, ranked, k):
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    rec, ndcg = [], []
    for truth, row_ in zip(ground_truth, ranked):
        ts = set(truth)
        hits = np.fromiter((i in ts for i in row_[:k]), np.float32, k)
        rec.append(hits.sum() / len(ts))
        idcg = disc[:min(len(ts), k)].sum()
        ndcg.append(float((hits * disc).sum()) / idcg if idcg > 0 else 0.0)
    return float(np.mean(rec)), float(np.mean(ndcg))


# --- FIX #3: tail defined INSIDE the evaluated item universe ------------------
# The old definition (item_pop <= median over all 3883 items) is empty by
# construction under EVAL_ON_BALANCED: the balanced set retains only items with
# >= 67 held-out interactions, every one of which is far above median. The
# metric returned a hardcoded 0.0 no matter what the model did.
def tail_items_for(eval_recs, pop, frac=TAIL_FRACTION):
    universe = np.array(sorted({i for v in eval_recs.values() for i in v}), dtype=np.int64)
    if universe.size == 0:
        return set(), 0.0
    pops = pop[universe]
    cutoff = float(np.quantile(pops, frac))
    return set(universe[pops <= cutoff].tolist()), cutoff


@torch.no_grad()
def evaluate(e_u, e_i, eval_recs, exclude_recs, pop, ks=TOP_KS, tail_set=None):
    users = sorted(u for u, v in eval_recs.items() if v)
    max_k = max(ks)
    ranked_all = []
    for s in range(0, len(users), 512):
        batch = users[s:s + 512]
        scores = e_u[batch] @ e_i.T
        for r, u in enumerate(batch):
            ex = exclude_recs.get(u, [])
            if ex:
                scores[r, torch.tensor(ex, dtype=torch.long, device=scores.device)] = -torch.inf
        ranked_all.append(torch.topk(scores, k=max_k, dim=1).indices.cpu().numpy())
    ranked = np.concatenate(ranked_all, 0)
    truth = [eval_recs[u] for u in users]

    out = {}
    for k in ks:
        r, n = recall_ndcg(truth, ranked, k)
        out[k] = {'recall': r, 'ndcg': n, 'arp': float(pop[ranked[:, :k]].mean())}

    if tail_set is None:
        tail_set, _ = tail_items_for(eval_recs, pop)
    tt = [[i for i in t if i in tail_set] for t in truth]
    keep = [j for j, t in enumerate(tt) if t]
    out['tail_recall@50'] = (recall_ndcg([tt[j] for j in keep], ranked[keep], 50)[0]
                             if keep else float('nan'))
    out['tail_users'] = len(keep)
    return out


class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, dim=LATENT_DIM):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, dim)
        self.item_embedding = nn.Embedding(num_items, dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def readout(self, adj):
        return propagate(adj, self.user_embedding.weight, self.item_embedding.weight)


def item_popularity_from(train_recs, num_items):
    c = np.zeros(num_items, dtype=np.float32)
    for items in train_recs.values():
        c[np.asarray(items, dtype=np.int64)] += 1.0
    return c / max(float(c.max()), 1.0)


# --- backbone checkpoint reuse ----------------------------------------------
# If a compatible lightgcn-ml1m.pt exists (this session's CKPT_DIR, or any attached
# /kaggle/input dataset), load it and SKIP training. Otherwise train and save one.
# "Compatible" = same seed, same train split (SHA-1 of every training interaction),
# same embedding dim / layers. A mismatched file is reported and ignored.
import hashlib
CKPT_NAME  = 'lightgcn-ml1m.pt'
CKPT_PATH  = os.path.join(CKPT_DIR, CKPT_NAME)
REQUIRE_CKPT              = True    # PPD must start from the BPSD backbone; refuse to retrain
FORCE_RETRAIN_BACKBONE     = False   # True -> ignore every saved checkpoint, train from scratch
ALLOW_UNFINGERPRINTED_CKPT = True    # accept an OLD-format .pt (weights only) if tensor shapes match;
                                     # it is then re-saved with a fingerprint (if SAVE_CHECKPOINTS)
_FP_CACHE = {}


def _backbone_fingerprint():
    if not _FP_CACHE:
        h = hashlib.sha1()
        for u in sorted(train_records):
            h.update(f'{u}:{",".join(map(str, sorted(train_records[u])))};'.encode())
        _FP_CACHE.update({'seed': SEED, 'num_users': NUM_USERS, 'num_items': NUM_ITEMS,
                          'train_inter': int(sum(map(len, train_records.values()))),
                          'train_sha1': h.hexdigest(),
                          'latent_dim': LATENT_DIM, 'n_layers': N_LAYERS})
    return dict(_FP_CACHE)


def _try_load_backbone_ckpt():
    """-> (state_dict, best_epoch, path, fingerprinted); (None, -1, None, False) if nothing usable."""
    if FORCE_RETRAIN_BACKBONE:
        print('[ckpt] FORCE_RETRAIN_BACKBONE=True -> training from scratch')
        return None, -1, None, False
    want = _backbone_fingerprint()
    cands = [CKPT_PATH]
    if os.path.isdir('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if CKPT_NAME in files:
                cands.append(os.path.join(root, CKPT_NAME))
    seen = set()
    for p in cands:
        p = os.path.abspath(p)
        if p in seen or not os.path.isfile(p):
            continue
        seen.add(p)
        try:
            try:
                ck = torch.load(p, map_location='cpu', weights_only=False)
            except TypeError:                                   # older torch: no weights_only arg
                ck = torch.load(p, map_location='cpu')
            state = ck['model_state_dict']
            ok = (tuple(state['user_embedding.weight'].shape) == (NUM_USERS, LATENT_DIM)
                  and tuple(state['item_embedding.weight'].shape) == (NUM_ITEMS, LATENT_DIM))
        except Exception as e:
            print(f'[ckpt] {p}: unreadable ({type(e).__name__}: {e}) -> skipped')
            continue
        if not ok:
            print(f'[ckpt] {p}: tensor shapes do not match this dataset/LATENT_DIM -> skipped')
            continue
        fp = ck.get('fingerprint')
        if fp is None:
            if not ALLOW_UNFINGERPRINTED_CKPT:
                print(f'[ckpt] {p}: old-format checkpoint, ALLOW_UNFINGERPRINTED_CKPT=False -> skipped')
                continue
            print(f'[ckpt] {p}: OLD-format checkpoint (no fingerprint); shapes match -> using it.\n'
                  '       Its split cannot be verified; it is valid only if it was trained with this '
                  'notebook\'s current SEED / split flags. Check the baseline numbers below.')
            return state, int(ck.get('epoch', -1)), p, False
        if fp != want:
            bad = [k for k in want if fp.get(k) != want[k]]
            print(f'[ckpt] {p}: fingerprint mismatch on {bad} -> skipped')
            continue
        print(f'[ckpt] reusing {p} (best epoch {int(ck.get("epoch", -1)) + 1}) - backbone training skipped')
        return state, int(ck.get('epoch', -1)), p, True
    print('[ckpt] no usable checkpoint found -> training backbone from scratch')
    return None, -1, None, False


def train_backbone():
    """Trains LightGCN from random init. Best state is kept IN MEMORY."""
    set_seed(SEED)
    adj = build_sparse_adj(train_records, NUM_USERS, NUM_ITEMS).to(DEVICE)
    model = LightGCN(NUM_USERS, NUM_ITEMS).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    pop = item_popularity_from(train_records, NUM_ITEMS)
    tail_val, cut_v = tail_items_for(val_records, pop)
    tail_test, cut_t = tail_items_for(test_records, pop)
    print(f'[tail] val: {len(tail_val)} items below pop {cut_v:.4f} | '
          f'test: {len(tail_test)} items below pop {cut_t:.4f}')

    ck_state, ck_ep, ck_path, ck_fp = _try_load_backbone_ckpt()
    if ck_state is None and REQUIRE_CKPT:
        raise FileNotFoundError(
            f'No compatible {CKPT_NAME} found. Attach the BPSD notebook output (the one containing '
            f'checkpoints/{CKPT_NAME}) via "+ Add Input", or set REQUIRE_CKPT = False to train a NEW '
            'backbone (then PPD and BPSD would NOT share a backbone).')
    best, best_ep, best_state = -np.inf, -1, None
    n_epochs = EPOCHS
    if ck_state is not None:                      # reuse saved weights: skip the training loop
        best_state, best_ep, n_epochs = ck_state, ck_ep, 0
    for epoch in range(n_epochs):
        model.train(); t0 = time.time()
        u, p, n = sample_triplets(train_records, NUM_ITEMS)
        perm = torch.randperm(len(u)); u, p, n = u[perm], p[perm], n[perm]
        tot, nb = 0.0, 0
        for s in range(0, len(u), BATCH_SIZE):
            ub  = u[s:s+BATCH_SIZE].to(DEVICE)
            pb  = p[s:s+BATCH_SIZE].to(DEVICE)
            nb_ = n[s:s+BATCH_SIZE].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            eu, ei = model.readout(adj)
            pos = (eu[ub] * ei[pb]).sum(1)
            neg = (eu[ub] * ei[nb_]).sum(1)
            rank = F.softplus(neg - pos).mean()
            ego = (model.user_embedding(ub).pow(2).sum()
                   + model.item_embedding(pb).pow(2).sum()
                   + model.item_embedding(nb_).pow(2).sum()) / (2.0 * len(ub))
            loss = rank + REG_WEIGHT * ego
            loss.backward(); opt.step()
            tot += loss.item(); nb += 1

        model.eval()
        with torch.no_grad():
            eu, ei = model.readout(adj)
            m = evaluate(eu, ei, val_records, EXCL_VAL, pop, tail_set=tail_val)
        if m[50]['ndcg'] > best:
            best, best_ep = m[50]['ndcg'], epoch
            best_state = copy.deepcopy(model.state_dict())
        print(f'Epoch [{epoch+1}/{EPOCHS}] Loss {tot/nb:.4f} '
              f'Recall@50 {m[50]["recall"]:.4f} NDCG@50 {m[50]["ndcg"]:.4f} '
              f'ARP@50 {m[50]["arp"]:.4f} {time.time()-t0:.1f}s')
        if epoch - best_ep >= PATIENCE:
            print(f'Early stop at {epoch+1}; best epoch {best_ep+1}'); break

    model.load_state_dict(best_state)
    model.eval()
    _need_save = (ck_path is None or not ck_fp
                  or os.path.abspath(ck_path) != os.path.abspath(CKPT_PATH))
    if SAVE_CHECKPOINTS and _need_save:           # trained now, old-format, or came from /kaggle/input
        torch.save({'epoch': best_ep,
                    'model_state_dict': {k: v.detach().cpu() for k, v in best_state.items()},
                    'fingerprint': _backbone_fingerprint()}, CKPT_PATH)
        print(f'[ckpt] saved -> {CKPT_PATH}')
    with torch.no_grad():
        eu, ei = model.readout(adj)
        m = evaluate(eu, ei, test_records, EXCL_TEST, pop, tail_set=tail_test)
    print('\n--- BASELINE LightGCN (test) ---')
    for k in TOP_KS:
        print(f'Recall@{k} {m[k]["recall"]:.4f}  NDCG@{k} {m[k]["ndcg"]:.4f}  '
              f'ARP@{k} {m[k]["arp"]:.4f}')
    print(f'Tail Recall@50 {m["tail_recall@50"]:.4f} '
          f'(over {m["tail_users"]} users with >=1 tail target)')
    return model, adj, pop, m


backbone, sparse_adj, item_pop, baseline_metrics = train_backbone()
TAIL_VAL,  _ = tail_items_for(val_records,  item_pop)
TAIL_TEST, _ = tail_items_for(test_records, item_pop)

/tmp/ipykernel_22/3127556707.py:24: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(


[tail] val: 151 items below pop 0.1454 | test: 149 items below pop 0.1466
[ckpt] reusing /kaggle/input/models/tejasdeshmukh001/lightgcn-ml1m-pt/pytorch/default/1/lightgcn-ml1m.pt (best epoch 437) - backbone training skipped

--- BASELINE LightGCN (test) ---
Recall@20 0.2608  NDCG@20 0.2245  ARP@20 0.3407
Recall@50 0.4441  NDCG@50 0.3001  ARP@50 0.2781
Recall@100 0.6169  NDCG@100 0.3588  ARP@100 0.2343
Tail Recall@50 0.2748 (over 3201 users with >=1 tail target)


## PPD implementation notes

PPD is defined by **Algorithm 1** of the paper in three steps:

1. **Popularity estimation** (Eqs. 5–7) — for every *training* interaction
   $(u,i)$, compute a popularity score
   $b_{ui} = p_i - r_{ui}$, where $p_i$ is the *global preference* of item
   $i$ (its average similarity to all users) and $r_{ui}$ is user $u$'s
   *personalized preference* for item $i$ (similarity to $u$'s own history,
   discounted by a popularity-penalty term controlled by $\beta$). Both
   $p_i$ and $r_{ui}$ are min-max normalized to $[0,1]$ before differencing
   (paper, Appendix B.2), and we compute them on the backbone's **stable,
   post-propagation** embeddings, exactly as the paper specifies.
2. **Popularity direction construction** (Eqs. 8–10) — for every user/item
   node, build a *popularity centroid* (neighbors weighted by $b_{ui}$) and a
   *preference centroid* (neighbors weighted by $1-b_{ui}$) from the
   backbone's **layer-0** embeddings, then take
   $\mathbf d_{pop}(v) = \bar{\mathbf e}_{pop}(v) - \phi\,\bar{\mathbf e}_{pref}(v)$.
3. **Debiasing via projection** (Eqs. 11–12) — subtract the projection of
   each layer-0 embedding onto its own popularity direction, then
   **re-propagate** the debiased layer-0 embeddings through the *same*
   normalized adjacency used to train the backbone. No retraining is ever
   performed — PPD is purely post-hoc.

Hyperparameters (paper, Appendix B.2), swept below on the same grid used in
the paper, selected by NDCG@50 on the balanced validation split (the same
selection rule your own BCPS sweep in CELL 6 uses):

* $\beta \in \{0.0, 0.1, 0.2, 0.3\}$ — popularity penalty inside $r_{ui}$.
* $\phi \in \{0.0, 0.25, 0.50, 0.75, 1.0\}$ — preference-centroid weight in
  the popularity direction.
* $\epsilon = 10^{-8}$ — denominator guard in the centroid formulas.

We also reproduce the paper's **RQ5 ablation** ("PPD w/o-PP"), which drops
the personalized-preference term and uses $b_{ui} = p_i$ directly.


In [3]:
# ============================================================================
# CELL PPD-1 — PPD: POST-HOC POPULARITY DEBIASING
#   Md Aminul Islam, Elena Zheleva, Ren Wang. "Post-hoc Popularity Bias
#   Correction in GNN-based Collaborative Filtering." WWW'26 (arXiv:2510.12959)
#
# Runs on top of the SAME pre-trained LightGCN backbone (`backbone`,
# `sparse_adj`) and the SAME train/val/test splits, exclusion sets, item
# popularity vector and evaluation protocol built in CELL 1 / CELL 2 above.
# PPD does NOT retrain the backbone: it only edits the layer-0 (l=0)
# embeddings post-hoc and re-propagates them through the already-built
# normalized adjacency `sparse_adj`, using the SAME `propagate()` function
# used to train/evaluate the backbone, so the comparison against your own
# post-hoc BCPS method is apples-to-apples (same backbone, same splits, same
# metrics, same masking).
#
# ---------------------------------------------------------------------------
# Paper recap (Sec. 4, Alg. 1):
#   1) Popularity estimation (Eqs. 5-7), computed on the STABLE, POST-
#      PROPAGATION backbone embeddings (paper: "we compute the popularity
#      score for each interaction using the stable user and item embeddings
#      learned by the pre-trained model"):
#         p_i   = (1/|U|) * sum_u sim(e_u, e_i)                       (5)
#         r_ui  = (1/|N_u|) * sum_{j in N_u} [sim(e_i,e_j) - beta*p_i*p_j]  (6)
#         b_ui  = p_i - r_ui                                          (7)
#      sim(.,.) = dot product (paper, Appendix B.2). p_i and r_ui are
#      min-max normalized to [0,1] before differencing (paper, Appendix B.2).
#   2) Popularity direction construction (Eqs. 8-10), computed on the
#      LAYER-0 (pre-propagation, "l=0") embeddings:
#         e_pop(v)  = weighted centroid of v's neighbours, weight = b_ui
#         e_pref(v) = weighted centroid of v's neighbours, weight = 1-b_ui
#         d_pop(v)  = e_pop(v) - phi * e_pref(v)
#   3) Debiasing via projection (Eqs. 11-12), applied at layer 0, THEN
#      propagated through all L layers of the GCN so unbiased signal flows
#      to every layer:
#         e~_v(0) = e_v(0) - Proj_{d_pop(v)}(e_v(0))
#
# Hyperparameters (paper, Appendix B.2): beta in {0.0,0.1,0.2,0.3} (popularity
# penalty in r_ui), phi in {0.0,0.25,0.50,0.75,1.0} (preference-centroid
# weight in d_pop), epsilon = 1e-8 (denominator guard in Eqs. 8-9).
# ============================================================================
PPD_EPS = 1e-8


def ppd_build_edge_lists(train_recs, device):
    """Flatten the training interaction set E = {(u,i)} into parallel
    (edge_u, edge_i) index tensors, exactly the interactions PPD is defined
    over (Alg. 1, line 5: 'for each interaction (u,i) in E')."""
    us, it = [], []
    for u, items in train_recs.items():
        us.extend([u] * len(items))
        it.extend(items)
    edge_u = torch.tensor(us, dtype=torch.long, device=device)
    edge_i = torch.tensor(it, dtype=torch.long, device=device)
    return edge_u, edge_i


def _minmax(x):
    return (x - x.min()) / (x.max() - x.min() + PPD_EPS)


def ppd_popularity_scores(eu_stable, ei_stable, edge_u, edge_i, num_users, beta):
    """Eqs. (5)-(7): per-interaction popularity score b_ui.

    eu_stable, ei_stable: the backbone's STABLE (post-propagation, final)
    user/item embeddings -- the same embeddings used for scoring
    (y_hat_ui = e_u^T e_i) in the pre-trained model.
    Returns (p_i_norm[num_items], b_edge[num_edges]) with both terms
    min-max normalized to [0,1] (paper, Appendix B.2), and b_edge additionally
    clamped to [0,1]: p_i_norm, r_ui_norm both lie in [0,1] so their raw
    difference lies in [-1,1]; a negative value means the interaction is
    explained by personal preference rather than popularity, which is exactly
    the b_ui -> 0 (=> preference weight 1-b_ui -> 1) limit the paper assigns
    to preference-driven interactions, so we floor at 0 rather than let a
    negative weight corrupt the convex centroids in Eqs. (8)-(9).
    """
    device = eu_stable.device

    # ---- Eq. (5): p_i = mean_u <e_u, e_i> = <e_i, mean_u e_u> ----------------
    u_bar = eu_stable.mean(dim=0)                           # [D]
    p_i = ei_stable @ u_bar                                  # [num_items]

    # ---- per-user aggregates needed to vectorize Eq. (6) over every edge -----
    # e_bar(u) = mean_{j in N_u} e_j ,  p_bar(u) = mean_{j in N_u} p_j
    D = ei_stable.shape[1]
    deg_u = torch.zeros(num_users, device=device).index_add_(
        0, edge_u, torch.ones_like(edge_u, dtype=torch.float32)).clamp(min=1.0)
    e_bar_u = torch.zeros(num_users, D, device=device).index_add_(
        0, edge_u, ei_stable[edge_i])
    e_bar_u = e_bar_u / deg_u.unsqueeze(1)
    p_bar_u = torch.zeros(num_users, device=device).index_add_(
        0, edge_u, p_i[edge_i]) / deg_u

    # ---- Eq. (6): r_ui = <e_i, e_bar(u)> - beta * p_i * p_bar(u) -------------
    r_ui = ((ei_stable[edge_i] * e_bar_u[edge_u]).sum(dim=1)
             - beta * p_i[edge_i] * p_bar_u[edge_u])

    # ---- normalize, then Eq. (7): b_ui = p_i - r_ui ---------------------------
    p_i_n = _minmax(p_i)
    r_ui_n = _minmax(r_ui)
    b_edge = (p_i_n[edge_i] - r_ui_n).clamp(min=0.0, max=1.0)

    return p_i_n, b_edge


def ppd_popularity_scores_wo_pp(eu_stable, ei_stable, edge_i):
    """RQ5 ablation, 'PPD w/o-PP': popularity score uses ONLY global
    preference p_i (min-max normalized), ignoring personalized preference
    entirely (paper Sec 5.2 / Table 5)."""
    u_bar = eu_stable.mean(dim=0)
    p_i = ei_stable @ u_bar
    p_i_n = _minmax(p_i)
    return p_i_n, p_i_n[edge_i].clamp(min=0.0, max=1.0)


def ppd_direction_vectors(eu0, ei0, edge_u, edge_i, b_edge, num_users, num_items, phi):
    """Eqs. (8)-(10): popularity/preference centroids (on LAYER-0 embeddings)
    and the resulting popularity direction vector, for every user and item
    node that has at least one training interaction. Nodes with none get a
    zero direction vector and are left untouched by the projection step."""
    device = eu0.device
    D = eu0.shape[1]
    w_pop = b_edge
    w_pref = 1.0 - b_edge

    # ---- user centroids, Eq. (8): aggregate over items j in N(u) -------------
    num_u = torch.zeros(num_users, device=device).index_add_(0, edge_u, w_pop)
    den_u = torch.zeros(num_users, device=device).index_add_(0, edge_u, w_pref)
    pop_u = torch.zeros(num_users, D, device=device).index_add_(
        0, edge_u, ei0[edge_i] * w_pop.unsqueeze(1))
    pref_u = torch.zeros(num_users, D, device=device).index_add_(
        0, edge_u, ei0[edge_i] * w_pref.unsqueeze(1))
    e_pop_u = pop_u / (num_u.unsqueeze(1) + PPD_EPS)
    e_pref_u = pref_u / (den_u.unsqueeze(1) + PPD_EPS)

    # ---- item centroids, Eq. (9): aggregate over users u in N(i) -------------
    num_i = torch.zeros(num_items, device=device).index_add_(0, edge_i, w_pop)
    den_i = torch.zeros(num_items, device=device).index_add_(0, edge_i, w_pref)
    pop_i = torch.zeros(num_items, D, device=device).index_add_(
        0, edge_i, eu0[edge_u] * w_pop.unsqueeze(1))
    pref_i = torch.zeros(num_items, D, device=device).index_add_(
        0, edge_i, eu0[edge_u] * w_pref.unsqueeze(1))
    e_pop_i = pop_i / (num_i.unsqueeze(1) + PPD_EPS)
    e_pref_i = pref_i / (den_i.unsqueeze(1) + PPD_EPS)

    # ---- Eq. (10): popularity direction vector --------------------------------
    d_u = e_pop_u - phi * e_pref_u
    d_i = e_pop_i - phi * e_pref_i
    return d_u, d_i


def ppd_project_remove(e0, d):
    """Eqs. (11)-(12): remove the popularity-aligned component of e0 via
    vector projection onto d, per node. Nodes with a (near-)zero-norm
    direction vector (e.g. no training interactions) are left unchanged,
    since the projection is undefined there."""
    dot = (e0 * d).sum(dim=1, keepdim=True)
    norm_sq = (d * d).sum(dim=1, keepdim=True)
    has_dir = norm_sq > PPD_EPS
    coeff = torch.where(has_dir, dot / norm_sq.clamp(min=PPD_EPS), torch.zeros_like(dot))
    return e0 - coeff * d


@torch.no_grad()
def run_ppd(beta, phi, with_personalized_preference=True,
            backbone=backbone, adj=sparse_adj, n_layers=N_LAYERS):
    """Full PPD pipeline (Algorithm 1): estimate b_ui on the pre-trained
    backbone's STABLE embeddings -> build per-node popularity direction on
    the LAYER-0 embeddings -> project-and-remove at layer 0 -> re-propagate
    through the SAME normalized adjacency used to train/evaluate the
    backbone -> evaluate with the notebook's own evaluate() (same exclusion
    sets, same tail split, same TOP_KS)."""
    backbone.eval()
    eu0 = backbone.user_embedding.weight.detach()
    ei0 = backbone.item_embedding.weight.detach()

    # stable (post-propagation) embeddings used only for popularity ESTIMATION
    eu_stable, ei_stable = propagate(adj, eu0, ei0, n_layers=n_layers)

    edge_u, edge_i = ppd_build_edge_lists(train_records, eu0.device)
    if with_personalized_preference:
        _, b_edge = ppd_popularity_scores(eu_stable, ei_stable, edge_u, edge_i,
                                          NUM_USERS, beta)
    else:
        _, b_edge = ppd_popularity_scores_wo_pp(eu_stable, ei_stable, edge_i)

    d_u, d_i = ppd_direction_vectors(eu0, ei0, edge_u, edge_i, b_edge,
                                     NUM_USERS, NUM_ITEMS, phi)

    eu0_tilde = ppd_project_remove(eu0, d_u)
    ei0_tilde = ppd_project_remove(ei0, d_i)

    # propagate the DEBIASED layer-0 embeddings through all L layers, exactly
    # as the paper prescribes ("...then propagate information...through all
    # subsequent layers of the GCN")
    eu_ppd, ei_ppd = propagate(adj, eu0_tilde, ei0_tilde, n_layers=n_layers)

    m_test = evaluate(eu_ppd, ei_ppd, test_records, EXCL_TEST, item_pop, tail_set=TAIL_TEST)
    m_val = evaluate(eu_ppd, ei_ppd, val_records, EXCL_VAL, item_pop, tail_set=TAIL_VAL)
    m_test['val_ndcg'] = m_val[50]['ndcg']
    return (eu_ppd, ei_ppd), m_test


In [4]:
# ============================================================================
# CELL PPD-2 — HYPERPARAMETER SWEEP, RQ5 ABLATION, FINAL COMPARISON TABLE
#
# Selection criterion: NDCG@50 on the (balanced) validation split -- the SAME
# selection criterion CELL 6 uses for your own BCPS sweep (`runs[k]['val_ndcg']`),
# so "PPD (best)" and "BCPS (best)" are chosen the same way before comparison.
#
# Grids follow the paper's own hyperparameter search (Appendix B.2):
#   beta in {0.0, 0.1, 0.2, 0.3}      (popularity penalty in r_ui, Eq. 6)
#   phi  in {0.0, 0.25, 0.5, 0.75, 1.0}   (preference-centroid weight, Eq. 10)
# ============================================================================
PPD_BETA_GRID = [0.0, 0.1, 0.2, 0.3]
PPD_PHI_GRID  = [0.0, 0.25, 0.5, 0.75, 1.0]


def ppd_row(name, m, ref=None):
    s = (f'{name:<32} R@50 {m[50]["recall"]:.4f}  N@50 {m[50]["ndcg"]:.4f}  '
         f'ARP@50 {m[50]["arp"]:.4f}  tail {m["tail_recall@50"]:.4f}')
    if ref is not None:
        s += (f'   vs base {100*(m[50]["ndcg"]/ref[50]["ndcg"]-1):+5.1f}% N '
              f'{100*(m[50]["arp"]/ref[50]["arp"]-1):+5.1f}% ARP')
    return s


print('=== PPD sweep (selection on balanced-val NDCG@50) ===')
ppd_runs = {}
for beta in PPD_BETA_GRID:
    for phi in PPD_PHI_GRID:
        _, m = run_ppd(beta=beta, phi=phi)
        ppd_runs[(beta, phi)] = m
        print(ppd_row(f'  beta={beta} phi={phi}', m, baseline_metrics))

PPD_BEST = max(ppd_runs, key=lambda k: ppd_runs[k]['val_ndcg'])
PB_BETA, PB_PHI = PPD_BEST
ppd_best_metrics = ppd_runs[PPD_BEST]
print(f'\n>>> PPD best by VAL NDCG@50: beta={PB_BETA}  phi={PB_PHI}')

# ---- RQ5 ablation: PPD vs PPD w/o personalized preference (paper Table 5) ---
_, ppd_wopp_metrics = run_ppd(beta=PB_BETA, phi=PB_PHI, with_personalized_preference=False)

print('\n=== PPD vs backbone vs RQ5 ablation ===')
print(ppd_row('0. LightGCN baseline', baseline_metrics))
print(ppd_row('1. PPD (best beta, phi)', ppd_best_metrics, baseline_metrics))
print(ppd_row('   PPD w/o-PP (RQ5)', ppd_wopp_metrics, baseline_metrics))

# ---- final head-to-head table: baseline vs PPD vs your own post-hoc method --
# (BCPS / `m_best` is populated once CELL 6 of your own pipeline has run;
# this block degrades gracefully if it hasn't.)
print('\n=== FINAL COMPARISON (test split) ===')
header = f'{"":<28}{"R@20":>8}{"N@20":>8}{"R@50":>8}{"N@50":>8}{"ARP@50":>9}{"tail@50":>9}'
print(header)


def _fmt(name, m):
    print(f'{name:<28}{m[20]["recall"]:>8.4f}{m[20]["ndcg"]:>8.4f}'
          f'{m[50]["recall"]:>8.4f}{m[50]["ndcg"]:>8.4f}'
          f'{m[50]["arp"]:>9.4f}{m["tail_recall@50"]:>9.4f}')


_fmt('LightGCN (no debiasing)', baseline_metrics)
_fmt(f'PPD (beta={PB_BETA}, phi={PB_PHI})', ppd_best_metrics)
try:
    _fmt('BCPS / your method (best)', m_best)   # defined in your CELL 6, if run
except NameError:
    print('(BCPS `m_best` not found in this kernel -- run your own pipeline '
          "cells first to populate it, or compare ppd_best_metrics against "
          "your saved results offline.)")

print('\nRelative gain over the SAME shared LightGCN backbone:')
gp = 100 * (ppd_best_metrics[50]['ndcg'] / baseline_metrics[50]['ndcg'] - 1)
print(f'  PPD  : {100*(ppd_best_metrics[50]["recall"]/baseline_metrics[50]["recall"]-1):+5.1f}% R@50  {gp:+5.1f}% N@50')
try:
    gb = 100 * (m_best[50]['ndcg'] / baseline_metrics[50]['ndcg'] - 1)
    print(f'  BCPS : {100*(m_best[50]["recall"]/baseline_metrics[50]["recall"]-1):+5.1f}% R@50  {gb:+5.1f}% N@50')
except NameError:
    pass

# ---- persist raw numbers for the paper's tables/plots -----------------------
import json as _json
ppd_export = {
    'grid': {'beta': PPD_BETA_GRID, 'phi': PPD_PHI_GRID},
    'best': {'beta': PB_BETA, 'phi': PB_PHI},
    'sweep': {f'{b}|{p}': {k: v for k, v in ppd_runs[(b, p)].items()
                            if k in TOP_KS + ['tail_recall@50', 'val_ndcg']}
              for (b, p) in ppd_runs},
    'best_metrics': {k: ppd_best_metrics[k] for k in TOP_KS + ['tail_recall@50', 'val_ndcg']},
    'wo_pp_metrics': {k: ppd_wopp_metrics[k] for k in TOP_KS + ['tail_recall@50', 'val_ndcg']},
    'baseline_metrics': {k: baseline_metrics[k] for k in TOP_KS + ['tail_recall@50']},
}
with open(os.path.join(OUT_ROOT, 'ppd_results.json'), 'w') as fh:
    _json.dump(ppd_export, fh, indent=2, default=str)
print(f"\n[saved] {os.path.join(OUT_ROOT, 'ppd_results.json')}")


=== PPD sweep (selection on balanced-val NDCG@50) ===
  beta=0.0 phi=0.0               R@50 0.1462  N@50 0.0935  ARP@50 0.2757  tail 0.0095   vs base -68.8% N  -0.8% ARP
  beta=0.0 phi=0.25              R@50 0.1773  N@50 0.1134  ARP@50 0.2968  tail 0.0152   vs base -62.2% N  +6.7% ARP
  beta=0.0 phi=0.5               R@50 0.2601  N@50 0.1671  ARP@50 0.2899  tail 0.0591   vs base -44.3% N  +4.3% ARP
  beta=0.0 phi=0.75              R@50 0.2918  N@50 0.1941  ARP@50 0.1836  tail 0.1811   vs base -35.3% N -34.0% ARP
  beta=0.0 phi=1.0               R@50 0.2299  N@50 0.1468  ARP@50 0.3550  tail 0.0223   vs base -51.1% N +27.7% ARP
  beta=0.1 phi=0.0               R@50 0.1481  N@50 0.0948  ARP@50 0.2755  tail 0.0098   vs base -68.4% N  -0.9% ARP
  beta=0.1 phi=0.25              R@50 0.1804  N@50 0.1156  ARP@50 0.2946  tail 0.0169   vs base -61.5% N  +6.0% ARP
  beta=0.1 phi=0.5               R@50 0.2602  N@50 0.1684  ARP@50 0.2778  tail 0.0650   vs base -43.9% N  -0.1% ARP
  beta=0.1 phi=0.7

## RQ5 / RQ7 on PPD
Run order: CELL 1 → CELL 2 (loads the BPSD checkpoint, never retrains) → PPD-1 → PPD-2 → RQ-COMMON → PPD-3 (RQ5) → PPD-4 (RQ7). Attach the BPSD notebook's output as an input so `lightgcn-ml1m.pt` can be found; attach the outputs `rq5_ppd_*.json` / `rq7_ppd_*.json` to the BPSD notebook.

In [5]:
# ============================================================================
# CELL RQ-COMMON — shared by the BPSD notebook and the PPD notebook (IDENTICAL text).
# Defines everything RQ5 (head/tail) and RQ7 (efficiency) measure with, so both
# notebooks score methods with exactly the same code:
#   * G            head/tail partition (+ a signature to verify both notebooks agree)
#   * topk_lists   top-k with the notebook's own masking rule (EXCL_TEST)
#   * eval_method  head/tail share + group Recall/NDCG for ANY score function
#   * Meter / time_inference   wall-clock, peak GPU memory, per-batch latency
# Needs Cells 1-2 of the respective notebook (splits, EXCL_TEST, recall_ndcg).
# ============================================================================
import gc, glob, hashlib, json, os, time
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt

for _n in ('DEVICE', 'NUM_USERS', 'NUM_ITEMS', 'OUT_ROOT', 'train_records', 'test_records',
           'EXCL_TEST', 'recall_ndcg'):
    assert _n in globals(), f'{_n} missing - run the data/backbone cells first'

DATASET_NAME  = globals().get('DATASET_NAME', 'ml-1m')
RQ5_HEAD_FRAC = 0.20          # head = top 20% of ALL items by TRAIN popularity
RQ5_K         = 20
RQ7_BATCH     = 512           # users per inference batch (= evaluation batch size)
RQ7_WARMUP, RQ7_ITERS = 10, 50


def _count(recs):
    arrs = [np.asarray(v, dtype=np.int64) for v in recs.values() if len(v)]
    return np.bincount(np.concatenate(arrs), minlength=NUM_ITEMS).astype(np.float64)


def build_groups(head_frac=RQ5_HEAD_FRAC):
    """Head/tail masks over the item catalogue, from TRAIN popularity (stable tie-break by id).
    'alt' = the same split taken INSIDE the evaluated universe (items that are test targets),
    needed because the balanced test set keeps only popular items."""
    train_cnt, test_cnt = _count(train_records), _count(test_records)
    head = np.zeros(NUM_ITEMS, dtype=bool)
    head[np.argsort(-train_cnt, kind='stable')[:int(np.ceil(head_frac * NUM_ITEMS))]] = True
    univ = test_cnt > 0
    ui = np.flatnonzero(univ)
    order = ui[np.argsort(-train_cnt[ui], kind='stable')]
    head_alt = np.zeros(NUM_ITEMS, dtype=bool)
    head_alt[order[:int(np.ceil(head_frac * len(ui)))]] = True
    sig = hashlib.sha1(train_cnt.astype(np.int64).tobytes() + test_cnt.astype(np.int64).tobytes()
                       + str(head_frac).encode()).hexdigest()
    return {'head': head, 'tail': ~head, 'head_alt': head_alt, 'tail_alt': univ & ~head_alt,
            'univ': univ, 'train_cnt': train_cnt, 'test_cnt': test_cnt, 'sig': sig,
            'ref_train_head': float(train_cnt[head].sum() / train_cnt.sum()),
            'ref_test_head': float(test_cnt[head].sum() / test_cnt.sum())}


G = build_groups()
users_eval = sorted(u for u, v in test_records.items() if v)
truth_eval = [test_records[u] for u in users_eval]


@torch.no_grad()
def topk_lists(score_fn, eval_recs, excl, k, bs=512):
    """Top-k ids per evaluated user; masks every known positive except the current targets
    (same rule and user order as evaluate()). score_fn(LongTensor[B] on DEVICE) -> [B, NUM_ITEMS]."""
    users = sorted(u for u, v in eval_recs.items() if v)
    out = []
    for s in range(0, len(users), bs):
        b = users[s:s + bs]
        sc = score_fn(torch.tensor(b, dtype=torch.long, device=DEVICE)).clone()
        for r, u in enumerate(b):
            ex = excl.get(u, [])
            if ex:
                sc[r, torch.tensor(ex, dtype=torch.long, device=DEVICE)] = -torch.inf
        out.append(torch.topk(sc, k=k, dim=1).indices.cpu().numpy())
    return users, np.concatenate(out, 0)


def group_metrics(ranked, truth, mask, k=RQ5_K):
    """Full top-k list scored against ONLY the targets inside `mask`; users without such a
    target are skipped. -> (recall, ndcg, n_users)"""
    tt = [[i for i in t if mask[i]] for t in truth]
    keep = [j for j, t in enumerate(tt) if t]
    if not keep:
        return float('nan'), float('nan'), 0
    r, n = recall_ndcg([tt[j] for j in keep], ranked[keep], k)
    return r, n, len(keep)


def eval_method(score_fn):
    users, ranked = topk_lists(score_fn, test_records, EXCL_TEST, RQ5_K)
    assert users == users_eval
    out = {'rec_head': float(G['head'][ranked].mean())}
    out['recall'], out['ndcg'] = recall_ndcg(truth_eval, ranked, RQ5_K)
    for tag, hm, tm in (('', G['head'], G['tail']), ('alt_', G['head_alt'], G['tail_alt'])):
        for g, mask in (('head', hm), ('tail', tm)):
            r, n, nu = group_metrics(ranked, truth_eval, mask)
            out[f'{tag}{g}_recall'], out[f'{tag}{g}_ndcg'], out[f'{tag}{g}_users'] = r, n, nu
    return out


# ---- efficiency measurement --------------------------------------------------------------------
def _sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


def device_name():
    return torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu'


class Meter:
    """with Meter() as m: ... -> m.seconds, m.peak_mb (absolute peak allocated), m.delta_mb
    (peak minus what was resident at entry = memory the section itself needed). NaN on CPU."""
    def __enter__(self):
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            self._base = torch.cuda.memory_allocated()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        _sync()
        self.seconds = time.perf_counter() - self._t0
        if DEVICE.type == 'cuda':
            peak = torch.cuda.max_memory_allocated()
            self.peak_mb, self.delta_mb = peak / 2 ** 20, (peak - self._base) / 2 ** 20
        else:
            self.peak_mb = self.delta_mb = float('nan')
        return False


_INFER = {}
def _infer_batch():
    if not _INFER:
        users = users_eval[:RQ7_BATCH]
        mask = torch.zeros(len(users), NUM_ITEMS, dtype=torch.bool, device=DEVICE)
        for r, u in enumerate(users):
            ex = EXCL_TEST.get(u, [])
            if ex:
                mask[r, torch.tensor(ex, dtype=torch.long, device=DEVICE)] = True
        _INFER['ub'] = torch.tensor(users, dtype=torch.long, device=DEVICE)
        _INFER['mask'] = mask
    return _INFER['ub'], _INFER['mask']


@torch.no_grad()
def time_inference(score_fn, k=20):
    """ms per batch of score -> mask known positives -> top-k. (median, p95) after warm-up."""
    ub, mask = _infer_batch()
    def run():
        return torch.topk(score_fn(ub).masked_fill(mask, -torch.inf), k=k, dim=1).indices
    for _ in range(RQ7_WARMUP):
        run()
    ts = []
    for _ in range(RQ7_ITERS):
        _sync(); t0 = time.perf_counter(); run(); _sync(); ts.append(1e3 * (time.perf_counter() - t0))
    return float(np.median(ts)), float(np.percentile(ts, 95))


def find_file(fname):
    """OUT_ROOT first, then any attached /kaggle/input dataset. -> path or None."""
    p = os.path.join(OUT_ROOT, fname)
    if os.path.isfile(p):
        return p
    if os.path.isdir('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if fname in files:
                return os.path.join(root, fname)
    return None


def _mean_std(x):
    x = np.asarray(x, dtype=np.float64)
    ok = np.isfinite(x).any()
    return (float(np.nanmean(x)) if ok else float('nan')), (float(np.nanstd(x)) if ok else float('nan'))


print(f'[rq-common] dataset={DATASET_NAME} device={device_name()} | head = {int(G["head"].sum())} items '
      f'(top {100 * RQ5_HEAD_FRAC:.0f}%) | interaction share in head: train {G["ref_train_head"]:.3f}, '
      f'balanced test {G["ref_test_head"]:.3f} | split signature {G["sig"][:10]}')
if 1.0 - G['ref_test_head'] < 0.01:
    print('[rq-common] NOTE: <1% of balanced-test interactions are tail items under the 20/80 split '
          '(the balanced set keeps only items with many held-out interactions). Tail metrics rest on '
          'very few targets; the evaluated-universe split (Table 6b) is the informative one.')


[rq-common] dataset=ml-1m device=Tesla T4 | head = 777 items (top 20%) | interaction share in head: train 0.732, balanced test 1.000 | split signature a61b738bcb
[rq-common] NOTE: <1% of balanced-test interactions are tail items under the 20/80 split (the balanced set keeps only items with many held-out interactions). Tail metrics rest on very few targets; the evaluated-universe split (Table 6b) is the informative one.


In [6]:
# ============================================================================
# CELL PPD-3 — RQ5 ON PPD: head/tail share + group Recall/NDCG@20  (run AFTER CELL PPD-2)
# Scores LightGCN and PPD(best beta, phi from PPD-2) with the SAME eval_method the BPSD
# notebook uses (RQ-COMMON cell), then saves rq5_ppd_<dataset>.json. The BPSD notebook's
# RQ5 cell loads that file as the "strongest baseline" and verifies that both notebooks
# used the same split (signature) and the same backbone (LightGCN numbers).
# ============================================================================
assert 'PB_BETA' in globals() and 'ppd_best_metrics' in globals(), 'run CELL PPD-2 first'
with torch.no_grad():
    EU_PROP, EI_PROP = propagate(sparse_adj, backbone.user_embedding.weight.detach(),
                                 backbone.item_embedding.weight.detach())
    (eu_ppd, ei_ppd), _m_ppd = run_ppd(beta=PB_BETA, phi=PB_PHI)

R5P = {'LightGCN': eval_method(lambda ub: EU_PROP[ub] @ EI_PROP.t()),
       'PPD':      eval_method(lambda ub: eu_ppd[ub] @ ei_ppd.t())}
for nme, ref in (('LightGCN', baseline_metrics), ('PPD', ppd_best_metrics)):
    d = abs(R5P[nme]['recall'] - ref[20]['recall'])
    print(f'[rq5] check {nme}: Recall@20 here {R5P[nme]["recall"]:.6f} vs earlier cells '
          f'{ref[20]["recall"]:.6f} -> {"OK" if d < 1e-4 else "MISMATCH - investigate"}')

print(f'\nRQ5 | dataset={DATASET_NAME} | PPD beta={PB_BETA} phi={PB_PHI} | top-{RQ5_K}')
print(f'train head share {G["ref_train_head"]:.4f} | balanced-test head share {G["ref_test_head"]:.4f}')
print(f'{"method":<10}{"head share":>11}{"HeadR":>9}{"HeadN":>9}{"TailR":>9}{"TailN":>9}'
      f'   | universe split: {"HeadR":>7}{"TailR":>8}')
for nme, r in R5P.items():
    print(f'{nme:<10}{r["rec_head"]:>11.4f}{r["head_recall"]:>9.4f}{r["head_ndcg"]:>9.4f}'
          f'{r["tail_recall"]:>9.4f}{r["tail_ndcg"]:>9.4f}   | {"":>15}{r["alt_head_recall"]:>7.4f}'
          f'{r["alt_tail_recall"]:>8.4f}')

_p = os.path.join(OUT_ROOT, f'rq5_ppd_{DATASET_NAME}.json')
json.dump({'dataset': DATASET_NAME,
           'config': {'head_frac': RQ5_HEAD_FRAC, 'k': RQ5_K, 'sig': G['sig'],
                      'beta': PB_BETA, 'phi': PB_PHI},
           'methods': R5P}, open(_p, 'w'), indent=1)
print(f'[rq5] saved -> {_p}   (attach it to the BPSD notebook as an input)')


[rq5] check LightGCN: Recall@20 here 0.260789 vs earlier cells 0.260789 -> OK
[rq5] check PPD: Recall@20 here 0.164202 vs earlier cells 0.164202 -> OK

RQ5 | dataset=ml-1m | PPD beta=0.0 phi=0.75 | top-20
train head share 0.7324 | balanced-test head share 1.0000
method     head share    HeadR    HeadN    TailR    TailN   | universe split:   HeadR   TailR
LightGCN       0.9226   0.2608   0.2245      nan      nan   |                 0.5475  0.1871
PPD            0.7061   0.1642   0.1416      nan      nan   |                 0.3525  0.1198
[rq5] saved -> /kaggle/working/bpsd_out/rq5_ppd_ml-1m.json   (attach it to the BPSD notebook as an input)


In [7]:
# ============================================================================
# CELL PPD-4 — RQ7 ON PPD: cost of the PPD post-hoc stage  (run AFTER CELL PPD-2)
#   single_pass : one PPD configuration, embeddings only (popularity estimation -> directions ->
#                 projection -> re-propagation); median of 3 warm passes
#   sweep       : the tuned pipeline = the 4x5 (beta, phi) grid, each config = pass + validation
#                 evaluation (what is needed to pick the best config; the test evaluations in
#                 PPD-2 are reporting, not part of the method's cost)
#   inference   : per-batch latency with the best config's embeddings cached
# Saves rq7_ppd_<dataset>.json; the BPSD notebook's RQ7 cell merges it into Table 7.
# Time/memory are only comparable with the BPSD numbers if both notebooks ran on the same
# hardware (device name is saved and compared).
# ============================================================================
assert 'PB_BETA' in globals() and 'run_ppd' in globals(), 'run CELL PPD-1 and PPD-2 first'


@torch.no_grad()
def ppd_embeddings(beta, phi):
    """run_ppd without the evaluation part (same maths, same functions)."""
    backbone.eval()
    eu0 = backbone.user_embedding.weight.detach()
    ei0 = backbone.item_embedding.weight.detach()
    eu_s, ei_s = propagate(sparse_adj, eu0, ei0, n_layers=N_LAYERS)
    edge_u, edge_i = ppd_build_edge_lists(train_records, eu0.device)
    _, b_edge = ppd_popularity_scores(eu_s, ei_s, edge_u, edge_i, NUM_USERS, beta)
    d_u, d_i = ppd_direction_vectors(eu0, ei0, edge_u, edge_i, b_edge, NUM_USERS, NUM_ITEMS, phi)
    return propagate(sparse_adj, ppd_project_remove(eu0, d_u), ppd_project_remove(ei0, d_i),
                     n_layers=N_LAYERS)


with torch.no_grad():
    (_eu_ref, _ei_ref), _ = run_ppd(beta=PB_BETA, phi=PB_PHI)
    _eu, _ei = ppd_embeddings(PB_BETA, PB_PHI)
    _ok = torch.allclose(_eu, _eu_ref, atol=1e-5) and torch.allclose(_ei, _ei_ref, atol=1e-5)
print(f'[rq7] ppd_embeddings == run_ppd embeddings: {"OK" if _ok else "MISMATCH - investigate"}')

_ts, _dl, _pk = [], [], []
for rep in range(4):                                    # rep 0 = warm-up, not counted
    with Meter() as m:
        ppd_embeddings(PB_BETA, PB_PHI)
    if rep:
        _ts.append(m.seconds); _dl.append(m.delta_mb); _pk.append(m.peak_mb)

with Meter() as msw:
    for _b in PPD_BETA_GRID:
        for _f in PPD_PHI_GRID:
            _e1, _e2 = ppd_embeddings(_b, _f)
            evaluate(_e1, _e2, val_records, EXCL_VAL, item_pop, tail_set=TAIL_VAL)
n_cfg = len(PPD_BETA_GRID) * len(PPD_PHI_GRID)

_inf, _inf95 = time_inference(lambda ub: _eu[ub] @ _ei.t())
R7P = {'dataset': DATASET_NAME, 'device': device_name(), 'config': {'beta': PB_BETA, 'phi': PB_PHI,
       'n_configs': n_cfg, 'batch': len(_infer_batch()[0])},
       'single_pass_s': float(np.median(_ts)), 'single_pass_peak_mb': float(np.max(_pk)),
       'single_pass_delta_mb': float(np.max(_dl)),
       'sweep_s': msw.seconds, 'sweep_peak_mb': msw.peak_mb, 'sweep_delta_mb': msw.delta_mb,
       'infer_ms': _inf, 'infer_p95_ms': _inf95}
print(f'[rq7] PPD on {R7P["device"]}: single pass {R7P["single_pass_s"]:.2f}s | '
      f'{n_cfg}-config tuned sweep {R7P["sweep_s"]:.1f}s ({R7P["sweep_s"] / 60:.2f} min) | '
      f'peak mem {R7P["sweep_peak_mb"]:.0f} MB (delta {R7P["sweep_delta_mb"]:.0f}) | '
      f'inference {R7P["infer_ms"]:.2f} ms/batch of {R7P["config"]["batch"]}')
_p = os.path.join(OUT_ROOT, f'rq7_ppd_{DATASET_NAME}.json')
json.dump(R7P, open(_p, 'w'), indent=1)
print(f'[rq7] saved -> {_p}   (attach it to the BPSD notebook as an input)')


[rq7] ppd_embeddings == run_ppd embeddings: OK
[rq7] PPD on Tesla T4: single pass 0.15s | 20-config tuned sweep 12.7s (0.21 min) | peak mem 441 MB (delta 400) | inference 1.02 ms/batch of 512
[rq7] saved -> /kaggle/working/bpsd_out/rq7_ppd_ml-1m.json   (attach it to the BPSD notebook as an input)
